# recs_029 — Stage 4 pivot: LLM explanation generation

`ranker_llm_local.py` (recs_028) showed a local LLM is a poor *reranker* for this catalog --
it lost decisively to v2a (NDCG@10 0.033 vs 0.097 on the same 200-example cohort). That's a
capability mismatch, not a broken pipeline: reranking needs the collaborative-filtering signal
baked into the trained `two_tower_v1`/v2a embeddings, which a zero-shot LLM reading truncated
text descriptions doesn't have access to.

Pivot: let the existing (good) pipeline pick the recommendation, and use the LLM for what it's
actually good at -- generating a grounded, human-readable explanation of *why* a real
recommendation makes sense. No promotion bar here (explanation quality is read, not scored);
this notebook is a qualitative spot-check.

Uses `StackedRecommender` (the actual shipped production pipeline: `two_tower_v1` retrieve ->
v2a rerank) so these are genuine recommendations, not cherry-picked or synthetic pairs.

In [1]:
import json
from pathlib import Path

from steam_review_ml.evaluation.candidate_text import build_candidate_text_lookup
from steam_review_ml.evaluation.example_cohort import load_retrieval_pools_jsonl
from steam_review_ml.recommender.llm_backends import LlamaCppBackend
from steam_review_ml.recommender.stacked_recommender import StackedRecommender

REPO_ROOT = Path.cwd().parent.parent
GGUF_PATH = REPO_ROOT / "artifacts/models/llm_local/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

rec = StackedRecommender.from_serve_config(repo_root=REPO_ROOT)
print(f"method_id={rec.method_id}, k_retrieval={rec.k_retrieval}, k_final={rec.k_final}")

backend = LlamaCppBackend(str(GGUF_PATH), n_gpu_layers=-1)

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-24 10:17:47.962882: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 10:17:48.044803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787581068.085871 1252135 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787581068.100188 1252135 cuda_bl

method_id=two_tower_v1_v2a_embed_query_logpop_blend, k_retrieval=100, k_final=10


## Get real recommendations for a handful of real reviews

Reusing 5 real (query_app_id, query_text) pairs from the `val_llm_mini_v1` frozen cohort --
genuine user reviews, not hand-written examples -- run through the actual production
`StackedRecommender.recommend()`.

In [2]:
POOLS_JSONL = REPO_ROOT / "artifacts/recs/eval_cache/val_llm_mini_v1/example_cohort.parquet"

import pandas as pd

cohort_df = pd.read_parquet(POOLS_JSONL)
sample_examples = cohort_df.sample(n=5, random_state=2026)[["query_app_id", "query_text"]].to_dict("records")

picks = []
for ex in sample_examples:
    query_app_id = int(ex["query_app_id"])
    hits = rec.recommend(ex["query_text"], query_app_id=query_app_id)
    top = hits.iloc[0]
    picks.append(
        {
            "query_app_id": query_app_id,
            "query_text": ex["query_text"],
            "rec_app_id": int(top["app_id"]),
            "rec_app_name": top.get("app_name", ""),
            "rec_score": float(top["score"]),
        }
    )

pd.DataFrame(picks)[["query_app_id", "rec_app_id", "rec_app_name", "rec_score"]]

,query_app_id,rec_app_id,rec_app_name,rec_score
0,252490,4000,Garry's Mod,0.958485
1,739630,105600,Terraria,0.988575
2,427520,8930,Sid Meier's Civilization V,0.965846
3,732810,212680,FTL: Faster Than Light,0.949816
4,306130,72850,The Elder Scrolls V: Skyrim,0.996160


In [3]:
sample_examples

[{'query_app_id': 252490,
  'query_text': 'Rust is a very hard but a very beautiful game'},
 {'query_app_id': 739630,
  'query_text': "  It's clearly not completed, afterall it's a Early Access. However at the current state the game is already fun and it has a lot of potential for future updates and eventual release."},
 {'query_app_id': 427520,
  'query_text': 'Ten out of ten, would decimate local populace of swarming insects with giant steel boot of industry again.'},
 {'query_app_id': 732810,
  'query_text': "OutRun 201X.  If that is all you're looking for out of your pick-up-and-play racer, this'll do you just fine.  Really enjoyed what time I've spent with it so far."},
 {'query_app_id': 306130,
  'query_text': 'Went into a dungeon, coping well, just about killed a group of enemies, 3 other players come down, steal the kills then demand 50 Gold for the rescue. Clippy but with ransom value simulator 2020'}]

## Generate explanations

Grounded in each game's actual IGDB text (`candidate_text.py`, already built for the ranker
spike) -- not the noisy review text, which is about the *query* game's playtime experience,
not a clean description suited for grounding an explanation.

In [4]:
all_app_ids = {p["query_app_id"] for p in picks} | {p["rec_app_id"] for p in picks}
game_texts = build_candidate_text_lookup(all_app_ids)

for p in picks:
    p["explanation"] = backend.generate_explanation(
        game_texts[p["query_app_id"]], game_texts[p["rec_app_id"]]
    )

for p in picks:
    print(f"Query app_id={p['query_app_id']} ({game_texts[p['query_app_id']].splitlines()[0]})")
    print(f"  -> Recommended: {p['rec_app_name']} (app_id={p['rec_app_id']}, score={p['rec_score']:.3f})")
    print(f"  Explanation: {p['explanation']}")
    print()

Query app_id=252490 (Rust)
  -> Recommended: Garry's Mod (app_id=4000, score=0.958)
  Explanation: We think you'll love Garry's Mod because it offers a similar sense of freedom and creativity as Rust, where you can build and experiment without any predefined goals. In Garry's Mod, you can use your imagination to create complex contraptions or simply have fun with silly characters, which we think is right up your alley if you enjoy the survival aspect of Rust by finding ways to outsmart its challenges.

Query app_id=739630 (Phasmophobia)
  -> Recommended: Terraria (app_id=105600, score=0.989)
  Explanation: We think you'll love Terraria because it offers a similar sense of exploration and discovery as Phasmophobia, where you get to uncover hidden secrets and treasures. In Terraria, you'll have the freedom to dig, build, and fight in a vast world, which is reminiscent of the investigative spirit of Phasmophobia's ghost-hunting gameplay. Plus, both games offer a high degree of customizati

In [5]:
all_app_ids

{4000, 8930, 72850, 105600, 212680, 252490, 306130, 427520, 732810, 739630}

In [6]:
print(game_texts[p["query_app_id"]])
print('-'*100)
print(game_texts[p["rec_app_id"]])

The Elder Scrolls Online

Every legend starts somewhere and in The Elder Scrolls Online, it starts with you. Write your story into a vibrant chapter of Tamriel’s distant past that takes place nearly 1,000 years before the iconic TES V: Skyrim, and discover a world steeped in adventure and possibility.

The game is set in the Second Era, in 2E 583, during a period of time known as the Interregnum. It was a period of time known for its political instability. Daedric prince Molag Bal has taken advantage of this instability to try and pull all of Tamriel into his realm of Coldharbour. Bal is doing so by sending devices called "Dark Anchors" into Tamriel. The Fighters Guild have taken it upon themselves to remove them.

The Tharn family, current rulers of Cyrodiil through Empress Regent Clivia Tharn, has made a pact with the King of Worms, who has agreed to supplement the Imperial's forces by resurrecting their soldiers. Secretly, Mannimarco is conspiring with Molag Bal, the Daedric Prince 